# Tema: Arquitectura Medallion

## Objetivos
Construir Bronze, Silver, cuarentena y Gold con contratos explícitos.

## Conceptos importantes para el examen
Bronze conserva origen; Silver limpia y deduplica; Gold sirve métricas. La arquitectura describe responsabilidades, no obliga a tres tecnologías distintas.

**Dificultad:** Intermedio · **Tiempo estimado:** 65 min.



Ejecuta la preparación una vez; después avanza celda a celda. Las soluciones modifican datos: úsalas tras tu intento. Para volver al estado inicial, ejecuta de nuevo la preparación completa (crea otro schema). No uses «Run all» para estudiar.

- [ ] Completado
- [ ] Necesito repasar
- [ ] Dominado

## Preparación y datos ficticios
Se necesita un notebook Python en Databricks con Spark y Unity Catalog. Solo se crean objetos en el schema de prácticas mostrado.

In [ ]:
# Cada ejecución de esta celda crea un schema NUEVO y aislado.
# El catálogo debe existir y permitir USE CATALOG y CREATE SCHEMA.
# Si no puedes crear schemas, pide uno de prácticas exclusivo y cambia SCHEMA.
import re
import uuid
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.window import Window

dbutils.widgets.text("catalog", spark.sql("SELECT current_catalog()").first()[0])
CATALOG = dbutils.widgets.get("catalog")
RUN_ID = uuid.uuid4().hex[:10]
SCHEMA = "dea_18_" + RUN_ID
def ident(value):
    return "`" + value.replace("`", "``") + "`"
spark.sql(f"USE CATALOG {ident(CATALOG)}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {ident(SCHEMA)}")
spark.sql(f"USE SCHEMA {ident(SCHEMA)}")
spark.sql("SET TIME ZONE 'UTC'")
print(f"Objetos de esta sesión: {CATALOG}.{SCHEMA}")
# No se borran automáticamente schemas, tablas ni checkpoints.


In [ ]:
raw_orders = spark.createDataFrame([(i, i % 4, str(i * 10), "2026-01-01") for i in range(1,19)] + [(1,1,"10","2026-01-01"), (19,1,"bad","2026-01-01"), (20,2,"-5","2026-01-01")], "order_id INT, customer_id INT, amount_raw STRING, date_raw STRING")
raw_orders.write.format("delta").mode("overwrite").saveAsTable("source_orders")

## PARTE 1 - EJEMPLOS GUIADOS

### 1. Bronze

In [ ]:
raw_orders.withColumn("ingested_at", F.current_timestamp()).write.format("delta").mode("overwrite").saveAsTable("bronze_orders")

### 2. Silver y cuarentena

In [ ]:
typed = spark.table("bronze_orders").withColumn("amount", F.expr("try_cast(amount_raw AS DECIMAL(12,2))")).withColumn("order_date", F.expr("try_cast(date_raw AS DATE)"))
rule = "order_id IS NOT NULL AND amount IS NOT NULL AND amount >= 0 AND order_date IS NOT NULL"
typed.filter(rule).dropDuplicates(["order_id"]).write.format("delta").mode("overwrite").saveAsTable("silver_orders")
typed.filter(f"NOT ({rule})").write.format("delta").mode("overwrite").saveAsTable("quarantine_orders")

### 3. Gold

In [ ]:
%sql
CREATE OR REPLACE TABLE gold_daily USING DELTA AS SELECT order_date, SUM(amount) revenue, COUNT(*) orders FROM silver_orders GROUP BY order_date;
SELECT * FROM gold_daily;

## PARTE 2 - EJERCICIOS
Resuelve todos antes de abrir las soluciones. Los ejercicios se realizan en orden y pueden usar resultados anteriores.

### EJERCICIO 1
Comprueba recuentos: Bronze 21, cuarentena 2, Silver 18; explica la fila restante.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 2
Añade reject_reason a cuarentena diferenciando importe inválido de negativo.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 3
Publica Gold por customer_id y verifica que la suma global coincide con gold_daily.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 4
Añade pedido 21 válido y recalcula Bronze/Silver/Gold sin duplicar los anteriores.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 5
Crea una vista de consumo y documenta cuándo preferirías tabla, materialized view o streaming table.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


## PARTE 3 - PISTAS
**Pista 1:** Una fila duplicada válida se elimina en Silver.

**Pista 2:** CASE WHEN sobre amount.

**Pista 3:** Agrupar no debe multiplicar importes.

**Pista 4:** Este ejercicio usa recomputación batch completa.

**Pista 5:** Compara latencia y mantenimiento.

## PARTE 4 - SOLUCIONES
**Detente aquí si todavía estás practicando.** Referencias completas para comparar después de resolver. Puedes plegar esta sección en Databricks.

### Solución 1

In [ ]:
assert spark.table("bronze_orders").count() == 21
assert spark.table("quarantine_orders").count() == 2
assert spark.table("silver_orders").count() == 18

### Solución 2

In [ ]:
%sql
CREATE OR REPLACE TABLE quarantine_labeled USING DELTA AS
SELECT *, CASE WHEN amount IS NULL THEN 'invalid_amount' WHEN amount < 0 THEN 'negative_amount' ELSE 'invalid_key_or_date' END reject_reason FROM quarantine_orders;

### Solución 3

In [ ]:
spark.table("silver_orders").groupBy("customer_id").agg(F.sum("amount").alias("revenue")).write.format("delta").mode("overwrite").saveAsTable("gold_customers")
a = spark.table("gold_customers").agg(F.sum("revenue")).first()[0]
b = spark.table("gold_daily").agg(F.sum("revenue")).first()[0]
assert a == b == 1710

### Solución 4

In [ ]:
spark.sql("INSERT INTO source_orders VALUES (21, 1, '100', '2026-01-02')")
spark.table("source_orders").withColumn("ingested_at", F.current_timestamp()).write.format("delta").mode("overwrite").saveAsTable("bronze_orders")
typed = spark.table("bronze_orders").withColumn("amount", F.expr("try_cast(amount_raw AS DECIMAL(12,2))")).withColumn("order_date", F.expr("try_cast(date_raw AS DATE)"))
typed.filter(rule).dropDuplicates(["order_id"]).write.format("delta").mode("overwrite").saveAsTable("silver_orders")
spark.sql("CREATE OR REPLACE TABLE gold_daily USING DELTA AS SELECT order_date, SUM(amount) revenue, COUNT(*) orders FROM silver_orders GROUP BY order_date")
assert spark.table("silver_orders").count() == 19

### Solución 5

In [ ]:
spark.sql("CREATE OR REPLACE VIEW revenue_for_bi AS SELECT order_date, revenue FROM gold_daily")
display(spark.table("revenue_for_bi"))
# Tabla: actualización explícita por nuestro job.
# Vista: consulta guardada sobre Gold.
# MV: resultado mantenido por infraestructura de refresco.
# Streaming table: procesamiento incremental gobernado por pipeline.

## PARTE 5 - PREGUNTAS TIPO EXAMEN
Preguntas originales de práctica; no son preguntas oficiales.

### Pregunta 1
¿Dónde guardarías el payload sin limpiar para reproceso?

A. Gold

B. Bronze

C. Solo en un checkpoint

D. Solo en un dashboard

### Pregunta 2
¿Dónde ubicas validación y deduplicación?

A. En los permisos exclusivamente

B. En el nombre del catálogo

C. Silver

D. Solo en el cliente BI

### Pregunta 3
¿Qué evita una inflación de ingresos por duplicados?

A. Deduplicar con una clave de negocio antes de agregar

B. SUM DISTINCT siempre

C. Aumentar workers

D. Reparticionar aleatoriamente

### Respuestas y explicación
**1. B** — Bronze conserva la entrada.

**2. C** — Silver aplica contratos para consumo posterior.

**3. A** — Elimina repeticiones de eventos, no importes iguales legítimos.

## PARTE 6 - RETO FINAL
Añade productos y una dimensión de clientes. Construye Gold por ciudad y día, separando claves huérfanas, y demuestra que los joins no multiplican ingresos.

Anota tu decisión, implementa el código y muestra evidencias. No se incluye solución para este reto.

In [ ]:
# TU RETO: código y verificaciones
